In [1]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Acumuladores Parte 4").getOrCreate()
sc = spark.sparkContext

# Acumuladores en Apache Spark

Los **acumuladores** son variables compartidas entre los ejecutores que se utilizan principalmente para implementar contadores o sumas globales dentro de tu programa de Spark de manera eficiente.

A diferencia de las variables convencionales, los acumuladores tienen una restricción de diseño importante: los ejecutores solo tienen permisos de **escritura** (para agregar datos), mientras que únicamente el programa principal (*Driver*) tiene permisos de **lectura** (para ver el resultado final).

---

## Puntos clave a tener en cuenta

* **`SparkContext.accumulator(valor_inicial)`**: Es el método que se utiliza en el *Driver* para definir e inicializar la variable del acumulador.
* **El método `add()`**: Es la función que utilizan las tareas (*tasks*) dentro de los ejecutores para agregar o actualizar un valor en el acumulador.
* **La propiedad `.value`**: Es la propiedad que se utiliza para recuperar el valor acumulado. Cabe destacar que **solo el *Driver* puede llamar a esta propiedad** para ver el resultado.

---

## Ejemplo Rápido en PySpark

Imagina que quieres procesar un RDD y, al mismo tiempo, contar cuántos registros vacíos o erróneos contiene:

```python
# 1. Inicializamos el acumulador en el Driver con un valor inicial de 0
contador_errores = sc.accumulator(0)

rdd_datos = sc.parallelize(["datos válidos", "", "más datos", "", "registro final"])

def procesar_registro(linea):
    # 2. Si la línea está vacía, los ejecutores usan add() para sumar 1
    if linea == "":
        contador_errores.add(1)
    return linea.upper()

# Ejecutamos una acción para disparar el procesamiento
resultado = rdd_datos.map(procesar_registro).collect()

# 3. El Driver recupera el valor total acumulado usando .value
print(f"Total de registros vacíos detectados: {contador_errores.value}")
# Salida: Total de registros vacíos detectados: 2

In [2]:
acumulador = sc.accumulator(0)

In [3]:
rdd = sc.parallelize([2,4,6,8,10])

In [4]:
rdd.foreach(lambda x: acumulador.add(x))

In [5]:
print(acumulador.value)

30


In [26]:
rdd1 = sc.parallelize("Mi nombre es Francisco y me siento en manifestacion".split(' '))

In [27]:
acumulador1 = sc.accumulator(0)

In [28]:
rdd1.foreach(lambda x: acumulador1.add(1))

In [29]:
print(acumulador1.value)

9
